# Homework 3 Part 2

## Setup

### Imports

In [29]:
from pyspark.sql import SparkSession, Row
from pyspark.pandas import DataFrame
from pyspark.sql.functions import expr, col, lit, broadcast, hash, max, min, avg, count, first, struct, collect_list, size, when
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

### Create Spark Session

In [30]:
spark = SparkSession.builder.appName("Jupyter").getOrCreate()

### Disable Automatic Broadcast Join

### Update system configurations to enable bucket join and preserve data grouping

In [31]:
spark.conf.set('spark.sql.sources.v2.bucketing.enabled','true') # For bucket joins to work
spark.conf.set('spark.sql.iceberg.planning.preserve-data-grouping','true') # For bucket joins to work
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1") # Disable automatic broadcast join

## Actors Cumulative Table Design (Query from Homework 1, Week 1, Dimensional Data Modeling)

### First establish DDLs for each table.

Delete table (if needed)

In [32]:
%%sql

DROP TABLE bootcamp.actor_films;

AnalysisException: [TABLE_OR_VIEW_NOT_FOUND] The table or view `demo`.`bootcamp`.`actor_films` cannot be found. Verify the spelling and correctness of the schema and catalog.
If you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.
To tolerate the error on drop use DROP VIEW IF EXISTS or DROP TABLE IF EXISTS.

In [33]:
%%sql

DROP TABLE bootcamp.actors;

AnalysisException: [TABLE_OR_VIEW_NOT_FOUND] The table or view `demo`.`bootcamp`.`actors` cannot be found. Verify the spelling and correctness of the schema and catalog.
If you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.
To tolerate the error on drop use DROP VIEW IF EXISTS or DROP TABLE IF EXISTS.

Create actor films table

In [34]:
%%sql

CREATE TABLE IF NOT EXISTS bootcamp.actor_films (
    actor STRING,
    actorid STRING,
    film STRING,
    year INTEGER,
    votes INTEGER,
    rating DOUBLE,
    filmid STRING
)
USING iceberg;

++
||
++
++

In [35]:
%%sql

CREATE TABLE IF NOT EXISTS bootcamp.actors (
    actor STRING,
    year INTEGER,
    quality_class STRING,
    is_active BOOLEAN,
    films ARRAY<STRUCT<film STRING, votes INTEGER, rating DOUBLE, filmid STRING, year INTEGER>>
)
USING iceberg;

++
||
++
++

## Read actor_films.csv and write to Iceberg table

Read in actor_films.csv and cast data types

In [36]:
actor_films_schema = StructType([
    StructField("actor", StringType(), True),
    StructField("actorid", StringType(), True),
    StructField("film", StringType(), True),
    StructField("year", IntegerType(), True),
    StructField("votes", IntegerType(), True),
    StructField("rating", DoubleType(), True),
    StructField("filmid", StringType(), True)
])

actor_films_df = spark.read \
    .option("header", "true") \
    .schema(actor_films_schema) \
    .csv("/home/iceberg/data/actor_films.csv")

actor_films_df.head(5)

[Row(actor='Fred Astaire', actorid='nm0000001', film='Ghost Story', year=1981, votes=7731, rating=6.3, filmid='tt0082449'),
 Row(actor='Fred Astaire', actorid='nm0000001', film='The Purple Taxi', year=1977, votes=533, rating=6.6, filmid='tt0076851'),
 Row(actor='Fred Astaire', actorid='nm0000001', film='The Amazing Dobermans', year=1976, votes=369, rating=5.3, filmid='tt0074130'),
 Row(actor='Fred Astaire', actorid='nm0000001', film='The Towering Inferno', year=1974, votes=39888, rating=7.0, filmid='tt0072308'),
 Row(actor='Lauren Bacall', actorid='nm0000002', film='Ernest & Celestine', year=2012, votes=18793, rating=7.9, filmid='tt1816518')]

Write to Iceberg table

In [37]:
actor_films_df.writeTo("bootcamp.actor_films") \
    .using("iceberg") \
    .option("overwrite-mode", "static") \
    .tableProperty("write.format.default", "parquet") \
    .overwritePartitions()
af = actor_films_df.alias("af")

## Create actors table

In [38]:
years = [(y,) for y in range(1970, 2022)]  # 2021 inclusive
years_df = spark.createDataFrame(years, ["year"])
y = years_df.alias("y")
years_df.head(3)

[Row(year=1970), Row(year=1971), Row(year=1972)]

In [39]:
first_actor_year_df = actor_films_df.groupBy("actor").agg(min("year").alias("first_year"))
fay = first_actor_year_df.alias("fay")
first_actor_year_df.head(3)

[Row(actor='Laurence Olivier', first_year=1970),
 Row(actor='Nastassja Kinski', first_year=1975),
 Row(actor='Daniel Day-Lewis', first_year=1971)]

In [40]:
actors_and_years_df = fay \
    .join(broadcast(y), col("fay.first_year") <= col("y.year")) \
    .select("fay.actor", "y.year")
aay = actors_and_years_df.alias("aay")
actors_and_years_df.head(3)

[Row(actor='Laurence Olivier', year=1970),
 Row(actor='Laurence Olivier', year=1971),
 Row(actor='Laurence Olivier', year=1972)]

In [41]:
film_struct = StructType([
    StructField("film", StringType(), True),
    StructField("votes", IntegerType(), True),
    StructField("rating", DoubleType(), True),
    StructField("filmid", IntegerType(), True),
    StructField("year", IntegerType(), True),
])
film_struct

StructType([StructField('film', StringType(), True), StructField('votes', IntegerType(), True), StructField('rating', DoubleType(), True), StructField('filmid', IntegerType(), True), StructField('year', IntegerType(), True)])

In [42]:
windowed_df = aay \
    .join(af, on=col("aay.actor") == col("af.actor"), how="left") \
    .filter(col("aay.year") >= col("af.year")) \
    .groupBy("aay.actor", "aay.year") \
    .agg(
        collect_list(
            struct("af.film", "af.votes", "af.rating", "af.filmid", "af.year")
        ).alias("films"),
        avg("af.rating").alias("avg_rating")
    )
w = windowed_df.alias("w")
windowed_df.show(3)

+-----------+----+--------------------+-----------------+
|      actor|year|               films|       avg_rating|
+-----------+----+--------------------+-----------------+
|AJ Michalka|2009|[{The Lovely Bone...|              6.7|
|AJ Michalka|2010|[{Secretariat, 26...|             6.95|
|AJ Michalka|2011|[{Super 8, 340065...|6.966666666666666|
+-----------+----+--------------------+-----------------+
only showing top 3 rows



In [44]:
actors_ready_df = w \
    .select(
        w.actor,
        w.year,
        when(col("w.avg_rating") > 8, "star") \
        .when((col("w.avg_rating") > 7) & (col("w.avg_rating") <= 8), "good") \
        .when((col("w.avg_rating") > 6) & (col("w.avg_rating") <= 7), "average") \
        .otherwise("bad").alias("quality_class"),
        when(col("w.films").getItem(0).year == w.year, True).otherwise(False).alias("is_active"),
        w.films
    ) \
    .orderBy("w.actor", "w.year")
ar = actors_ready_df.alias("ar")
actors_ready_df.head(2)

[Row(actor='50 Cent', year=2005, quality_class='bad', is_active=True, films=[Row(film="Get Rich or Die Tryin'", votes=44370, rating=5.4, filmid='tt0430308', year=2005)]),
 Row(actor='50 Cent', year=2006, quality_class='bad', is_active=True, films=[Row(film='Home of the Brave', votes=10500, rating=5.6, filmid='tt0763840', year=2006), Row(film='Vengeance', votes=133, rating=3.5, filmid='tt0485920', year=2006), Row(film="Get Rich or Die Tryin'", votes=44370, rating=5.4, filmid='tt0430308', year=2005)])]

In [45]:
ar.writeTo("bootcamp.actors") \
    .using("iceberg") \
    .option("overwrite-mode", "static") \
    .tableProperty("write.format.default", "parquet") \
    .overwritePartitions()

In [47]:
%%sql

SELECT * FROM bootcamp.actors WHERE actors.actor = '50 Cent' AND actors.year <= 2006;

actor,year,quality_class,is_active,films
50 Cent,2005,bad,True,"[Row(film=""Get Rich or Die Tryin'"", votes=44370, rating=5.4, filmid='tt0430308', year=2005)]"
50 Cent,2006,bad,True,"[Row(film='Home of the Brave', votes=10500, rating=5.6, filmid='tt0763840', year=2006), Row(film='Vengeance', votes=133, rating=3.5, filmid='tt0485920', year=2006), Row(film=""Get Rich or Die Tryin'"", votes=44370, rating=5.4, filmid='tt0430308', year=2005)]"
